In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Image, display, Markdown
from scraper import fetch_website_contents, fetch_website_links
from utilities import usage_metrics

In [2]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY is not set")
elif api_key.startswith("sk-proj"):
    print("Valid OPENAI API key found")

Valid OPENAI API key found


In [3]:
MODEL = "gpt-5-nano"
openai = OpenAI()
COMPANY_NAME = "PPFAS"
URL = "https://amc.ppfas.com/index.php"

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of these links would be relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or a Careers/jobs page or a Products page.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url":"https://full.url/goes/here/about"},
        {"type": "carrers page", "url":"https://another.full.url/goes/here/careers"},
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f""""
    Here is the list of links on the website {url} -
    Please decide which of these links are relevant for a brochure about the company.
    Respond with the full https url in a JSON format.
    Do Not include Terms of Service, Privacy, email links.

    Links (some might be relative links):

    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
import json


def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content":link_system_prompt},
            {"role":"user", "content":get_links_user_prompt(url=url)}
        ],
        response_format={"type":"json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)

    print(f"Found {len(links['links'])} relevant links")    
    return links
# links = select_relevant_links(URL)


In [7]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url=url)
    relevant_links = select_relevant_links(url=url)
    result = f"## Landing Page:\n\n{contents}\n##Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n### Link: {link["type"]}\n"
        result += fetch_website_contents(link["url"])
    return result

In [8]:
brochure_system_prompt = """
You are an assistant that analyses the contents of several web pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.

Include details of company culture, customers and careers/jobs if the information is available. 
"""

In [9]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f""""
    You are looking at a company called {company_name}.
    Here are the contents of its most relevant webpages:
    Use this information to build a short brochure of the company in markdown without code blocks. \n\n
    """
    user_prompt += fetch_page_and_all_relevant_links(url=url)
    user_prompt = user_prompt[:5000]
    return user_prompt

In [10]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system", "content":brochure_system_prompt},
            {"role":"user", "content":get_brochure_user_prompt(company_name=company_name, url=url)}
        ]
    )
    return response.choices[0].message.content

response = create_brochure(COMPANY_NAME, URL)

Found 25 relevant links


In [11]:
display(Markdown(response))

# PPFAS: Value Investing for Long-Term Wealth

PPFAS Asset Management Private Limited builds wealth through disciplined, value-driven investing. As the sponsor of PPFAS Mutual Fund, the company emphasizes long-term thinking, investor education, and transparent communication.

## About PPFAS

- Founded mindset: Managing money using Value Investing principles.
- Status: Sponsor of PPFAS Mutual Fund; offers a range of investment services and wealth solutions.
- Location: Corporate office in Mumbai (81/82, 8th Floor, Sakhar Bhavan, Ramnath Goenka Marg, Nariman Point, Mumbai 400069, India).
- Contact: Tel +91 22 6140 6555 | Email [email protected]
- Online access: Invest Online and PPFAS SelfInvest app for mobile and web access.

## Our Philosophy

- Core approach: Value investing with a long-term perspective.
- Investment style: Dynamic asset allocation to balance growth and risk.
- Key fund objective (Parag Parikh Dynamic Asset Allocation Fund): Generate income and long-term capital appreciation by investing across equity, equity derivatives, and fixed income. Allocation between asset classes is managed dynamically to pursue upside while managing downside risk. Important: there are no guarantees of returns; the scheme aims to achieve its objective but does not assure or guarantee performance.
- Commitment to education: A strong emphasis on understanding behavioral finance, biases, and practical investing skills.

## Our Funds and Services

- PPFAS Mutual Fund: Positioned as a value-focused mutual fund for long-term investors.
- Parag Parikh Dynamic Asset Allocation Fund: Open-ended fund with dynamic asset allocation across equities and fixed income.
- PPFAS Wealth: Wealth management division offering personalized investment solutions.
- PPFAS GIFT: Inbound/outbound funds and Portfolio Management Services.
- Portfolio Management and advisory products: Cognito, Multi-Asset Non Discretionary Strategy, and other investment services.
- Online tools and resources: Net Asset Values, investment statements, and access via SelfInvest app.

## Education, Thought Leadership & Community

- Knowledge Centre: Resources to enhance financial literacy and understanding of investing.
- Tutorials and videos: Educational content including behavioral finance concepts and practical tutorials.
- Behavioral finance focus: Articles and materials on mental accounting, biases, and how to invest more effectively.
- Media presence: Updates and articles about PPFAS in the news and company materials.
- Financial Opportunities Forum: Monthly meetings in Mumbai to discuss investment opportunities and insights (subscribe or learn more).
- PPFAS TV: Updates and thought leadership from the firm.
- Founder’s legacy: Parag Parikh’s memoir “Timeless” and tributes honoring his contributions to value investing.

## Culture & People

- Culture centered on disciplined investing, intellectual rigor, and a commitment to investor education.
- Leadership emphasis: Founders and directors guided by values of transparency, long-term thinking, and prudent risk management.
- Careers: The company signals ongoing hiring through “We Are Hiring,” inviting talented individuals to join the team.

## Careers & Opportunities

- We are Hiring: Opportunities across various roles consistent with a research-driven, investor-focused organization.
- Careers page typically features roles in portfolio management, research, client service, and education/communications.

## Customers & How to Engage

- Primary audience: New and existing investors seeking long-term value investing solutions.
- Clients also include distributors and partners collaborating through the investor desk and other service channels.
- Investor resources: NAVs, statements, missed-call account statements, and other investor services through online portals and mobile apps.

## Quick Take for Prospective Customers, Investors & Recruits

- Invest with a philosophy of value and long-term growth.
- Benefit from dynamic asset allocation that aims to manage downside risk.
- Access robust education and thought leadership to support better investing decisions.
- Explore a spectrum of services from mutual funds to wealth management and gift/portfolio services.
- Engage with a company open to hiring and growing talent in a value-driven culture.

For more information or to start your journey with PPFAS, contact your relationship manager or visit the online portals and resources referenced above.

# Add Streaming

In [14]:
from IPython.display import update_display


def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role":"system", "content":brochure_system_prompt},
            {"role":"user", "content":get_brochure_user_prompt(company_name=company_name, url=url)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [15]:
stream_brochure(COMPANY_NAME, URL)

Found 12 relevant links


# PPFAS Mutual Fund Brochure

---

## About PPFAS Mutual Fund

**PPFAS Asset Management Pvt. Ltd.** is a trusted name in value-oriented long-term investing in India. Founded by Parag Parikh Financial Advisory Services Ltd. (PPFAS Ltd.) — a boutique investment advisory firm established in 1992 — PPFAS AMC has established itself as one of India’s earliest SEBI Registered Portfolio Management Service (PMS) providers.

### Our Mission  
To help clients achieve their long-term financial goals through prudent fund management rooted in Value Investing principles.

### Our Philosophy & Beliefs  
- We strongly adhere to the **'Law of the Farm'**: investments take time to mature, and attempts to hasten this process can be counterproductive. This philosophy underwrites our focus on relatively long holding periods.  
- Investing should be **simple and transparent**. We strive for simplicity in scheme design, investment processes, and operations.  
- We believe fund management is a **professional responsibility and fiduciary duty** rather than just a business. Our clients' interests always take priority.

---

## What Makes Us Different?

- **Value Investing at Core**: Our investment approach is discipline-driven, focusing on delivering long-term capital appreciation while managing downside risks.  
- **Dynamic Asset Allocation**: For example, our Parag Parikh Dynamic Asset Allocation Fund balances equity and fixed income dynamically to optimize returns and reduce risk.  
- **Client-centric Operations**: Emphasizing transparency, accountability, and straightforward communication with investors.  
- **Technology-enabled Access**: Investors can use our PPFAS SelfInvest app (mobile and web) for account management and transactions.  

---

## Customers & Community

Our primary customer base includes long-term investors looking for value and prudent fund management. We service both new and existing investors through multiple channels that provide insights, education, and seamless investment experiences.

---

## Company Culture & Careers

- We foster a **culture of discipline, simplicity, and professionalism** in fund management.  
- PPFAS AMC views investment management not just as a business but a trusted fiduciary relationship with clients.  
- Our team embraces transparency, patience, and a process-driven approach rooted in strong ethical values.  
- Career Opportunities are periodically posted on our website, inviting professionals who share our values of integrity and long-term focus.

---

## Contact Information

**Address:**  
81/82, 8th Floor, Sakhar Bhavan, Ramnath Goenka Marg,  
Mumbai, Maharashtra, 400069, India

**Telephone:** +91 22 6140 6555  
**Fax:** +91 22 6140 6590  
**Email:** [info@ppfas.com](mailto:info@ppfas.com)

---

## Explore Our Services

- Investment in Equities, Fixed Income, Equity Derivatives  
- Long-Term Capital Appreciation Strategies  
- Prudently Managed Mutual Fund Schemes  
- Debt Market Insights and Regular Account Statements  
- Robust digital platforms for investment management  

---

**PPFAS Mutual Fund** — Managing your money with patience, discipline, and integrity for long-term growth.  

For more information, visit: [https://www.ppfas.com](https://www.ppfas.com)